# Logic Study

In [51]:
from pathlib import Path
import pickle
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Configuration

### AIT-LDS V2

In [52]:
phase_map = {
    "benign": 0,
    "phase1": 1,
    "phase2": 2,
    "phase3": 3,
    "phase4": 4,
}

plot_labels = [
    "benign",
    "phase1",
    "phase2",
    "phase3",
    "phase4",
]

In [53]:
dataset = "aitv2"
scenario = "santos"
test_scenario = "santos"

sim_start = pd.Timestamp("2022-01-14 18:00")
attack_start = pd.Timestamp("2022-01-17 11:00")
attack_end   = pd.Timestamp("2022-01-17 12:00")

In [54]:
# Cross-scenario config
# scenario = "santos_fox"
# test_scenario = "fox"

# attack_start = pd.Timestamp("2022-01-18 11:59")
# attack_end   = pd.Timestamp("2022-01-18 13:15")

### DARPA 2000

In [55]:
# phase_map = {
#     "benign": 0,
#     "phase1": 1,
#     "phase2": 2,
#     "phase3": 3,
#     "phase4": 4,
#     "phase5": 5,
# }

# plot_labels = [
#     "benign",
#     "phase1",
#     "phase2",
#     "phase3",
#     "phase4",
#     "phase5",
# ]

In [56]:
# dataset = "darpa2000"
# scenario = "s1_inside"
# test_scenario = "s1_inside"

## Metrics

In [57]:
experiment = "logic_study"
experiment_dir = Path(f"../../experiments/{dataset}/{scenario}/{experiment}")
reports_dir = Path(f"../../reports/{dataset}/{scenario}/{experiment}")
reports_dir.mkdir(parents=True, exist_ok=True)

### Compute attack phase boundaries

In [58]:
df = pd.read_csv(
    f"../../data/interim/{dataset}/{test_scenario}/flows_labeled/all_flows_behavioral.csv"
)

df = df.sort_values("start_time").reset_index(drop=True)

In [59]:
if dataset == "darpa2000":
    sim_start = df["start_time"].min()
    print(f"Sim start time: {sim_start}")
    df["start_time"] = pd.to_datetime(df["start_time"], unit="s", origin=sim_start)
    # Convert start_time and end_time from seconds to datetime
    df["start_time_dt"] = pd.to_datetime(df["start_time"], unit="s")
    df["end_time_dt"] = pd.to_datetime(df["end_time"], unit="s")

    print(f"Full dataset time range: {df['start_time_dt'].min()} to {df['end_time_dt'].max()}")
    print("Total flows in combined dataset:", len(df))

In [60]:
df["start_time_dt"] = pd.to_datetime(df["start_time_dt"], errors="coerce")

phase_bounds = (
    df[df["phase"] != "benign"]
    .groupby('phase')['start_time_dt']
    .agg(['min', 'max'])
)

phase_starts = phase_bounds['min'].to_dict()

In [61]:
phase_bounds

,min,max
phase,,
0,2022-01-14 00:00:02.425734043,2022-01-17 23:59:53.819441080
1,2022-01-14 00:00:09.731323957,2022-01-16 07:16:04.975357056
2,2022-01-17 11:15:12.900707960,2022-01-17 11:22:44.917221069
3,2022-01-17 11:22:57.162303925,2022-01-17 11:24:14.067639112
4,2022-01-17 11:24:16.209572077,2022-01-17 11:57:27.041970015


### Compute temporal metrics

In [62]:
def is_causal_violation(row, phase_starts):
    pred_phase = row['y_pred']
    t = row['start_time_dt']

    if pred_phase == 0 or pred_phase == 1:
        return False
    
    prev_phase = pred_phase - 1
    
    if prev_phase in phase_starts:
        if t < phase_starts[prev_phase]:
            return True

    return False


def is_regression_violation(row, phase_starts):
    pred_phase = row['y_pred']
    t = row['start_time_dt']

    if pred_phase == 0:
        return False

    next_phase = pred_phase + 1

    if next_phase in phase_starts:
        if t >= phase_starts[next_phase]:
            return True

    return False


def temp_metrics(exp_df, f1, phase_starts):
    df = exp_df.copy()

    df["start_time_dt"] = pd.to_datetime(df["start_time_dt"], errors="coerce")
    df["end_time_dt"] = pd.to_datetime(df["end_time_dt"], errors="coerce")

    df['causal_violation'] = df.apply(
        is_causal_violation,
        axis=1,
        phase_starts=phase_starts
    )

    df['regression_violation'] = df.apply(
        is_regression_violation,
        axis=1,
        phase_starts=phase_starts
    )
    
    correct_df = df[df['phase'] == df['y_pred']]
    wrong_df = df[df['phase'] != df['y_pred']]
    causal_df = wrong_df[wrong_df['causal_violation']]
    regression_df = wrong_df[(~wrong_df['causal_violation']) & (wrong_df['regression_violation'])]

    num_samples = len(df)
    num_wrong = len(wrong_df)
    num_causal = len(causal_df)
    num_regression = len(regression_df)

    error_rate = num_wrong / num_samples if num_samples > 0 else 0  
    causal_rate = num_causal / num_wrong if num_wrong > 0 else 0
    regression_rate = num_regression / num_wrong if num_wrong > 0 else 0

    temp_score = f1 - 0.5 * causal_rate - 0.2 * regression_rate

    dataframes = {
        "correct": correct_df,
        "causal": causal_df,
        "regression": regression_df,
    }

    temp_metrics_dict = {
        "num_wrong": num_wrong,
        "num_causal": num_causal,
        "num_regression": num_regression,
        "error_rate": error_rate,
        "causal_rate": causal_rate,
        "regression_rate": regression_rate,
        "temp_score": temp_score
    }
    
    return dataframes, temp_metrics_dict


In [63]:
results = []
metrics_dict = {}
dataframes_dict = {}

### Kitsune

In [64]:
def compute_weighted_f1(metrics):
    total_support = 0
    weighted_f1_sum = 0

    for class_name, stats in metrics['per_class'].items():
        support = stats['TP'] + stats['FN']
        total_support += support
        weighted_f1_sum += stats['f1'] * support

    weighted_f1 = weighted_f1_sum / total_support

    return weighted_f1

In [65]:
# --- Load metrics ---
metrics_dir = Path(f"../../experiments/{dataset}/{scenario}/kitsune")
file_paths = list(metrics_dir.iterdir())
    
for file_path in file_paths:
    if "metrics" not in file_path.name:
        continue
    
    with open(file_path) as f:
        metrics = json.load(f)

    weighted_f1 = compute_weighted_f1(metrics)
    print(f"Experiment: {file_path}, Weighted F1: {weighted_f1}")

Experiment: ../../experiments/aitv2/santos/kitsune/metrics_20260614_131804.json, Weighted F1: 0.9376716071903528


### DPL Models

In [66]:
dpl_experiments = {}

In [67]:
# --- Load metrics ---
metrics_dir = Path(f"../../experiments/{dataset}/{scenario}/{experiment}/deepproblog/metrics")
file_paths = list(metrics_dir.iterdir())
    
for file_path in file_paths:
    experiment_name = str(file_path.stem)

    # Metrics
    data = np.load(file_path, allow_pickle=True)
    print(experiment_name)
    dpl_experiments[experiment_name] = {
        "confusion_matrix": data["confusion_matrix"],
        "classes": data["classes"].tolist(),
        "metrics": data["metrics"].item(),
    }

ait_temp_context_pretrained_base_w100_full_0.001lr_20260605_215320
ait_temp_pretrained_base_w100_full_0.001lr_20260522_230036
ait_temp_endtoend_base_w100_full_0.001lr_20260606_015802
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260613_093832
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260610_130712
ait_temp_context_endtoend_base_w100_full_0.001lr_20260605_124548
ait_temp_context_pretrained_base_w100_full_0.001lr_20260522_172417
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260611_094556
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260524_230019
ait_temp_context_endtoend_base_w100_full_0.001lr_20260521_235742
ait_temp_endtoend_base_w100_full_0.001lr_20260606_153534
ait_temp_context_pretrained_base_w100_full_0.001lr_20260605_203228
ait_temp_pretrained_base_w100_full_0.001lr_20260609_054458
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260611_083021
ait_temp_context_pretrained_base_w100_full_0.001lr_20260606_003729
ait_temp_baseline_endtoe

In [68]:
errors_dir = Path(f"../../experiments/{dataset}/{scenario}/{experiment}/deepproblog/errors")
file_paths = list(errors_dir.iterdir())
    
for file_path in file_paths:
    experiment_name = str(file_path.stem)

    with open(file_path, "r") as f:
        errors = json.load(f)
        
    print(experiment_name)
    dpl_experiments[experiment_name]["errors"] = errors

ait_temp_pretrained_base_w100_full_0.001lr_20260609_054458
ait_temp_context_pretrained_base_w100_full_0.001lr_20260605_215320
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260611_083021
ait_temp_endtoend_base_w100_full_0.001lr_20260523_232729
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260611_090758
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260524_230019
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260611_094556
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260524_223704
ait_temp_pretrained_base_w100_full_0.001lr_20260608_114051
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260613_093832
ait_temp_endtoend_base_w100_full_0.001lr_20260607_205213
ait_temp_context_endtoend_base_w100_full_0.001lr_20260605_150528
ait_temp_endtoend_base_w100_full_0.001lr_20260607_051456
ait_temp_endtoend_base_w100_full_0.001lr_20260606_153534
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260610_130712
ait_temp_baseline_endtoend_b

In [69]:
correct_dir = Path(f"../../experiments/{dataset}/{scenario}/{experiment}/deepproblog/correct")
file_paths = list(correct_dir.iterdir())
    
for file_path in file_paths:
    experiment_name = str(file_path.stem)

    with open(file_path, "r") as f:
        errors = json.load(f)
        
    print(experiment_name)
    dpl_experiments[experiment_name]["correct"] = errors

ait_temp_pretrained_base_w100_full_0.001lr_20260609_054458
ait_temp_context_pretrained_base_w100_full_0.001lr_20260605_215320
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260611_083021
ait_temp_endtoend_base_w100_full_0.001lr_20260523_232729
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260611_090758
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260524_230019
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260611_094556
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260524_223704
ait_temp_pretrained_base_w100_full_0.001lr_20260608_114051
ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260613_093832
ait_temp_endtoend_base_w100_full_0.001lr_20260607_205213
ait_temp_context_endtoend_base_w100_full_0.001lr_20260605_150528
ait_temp_endtoend_base_w100_full_0.001lr_20260607_051456
ait_temp_endtoend_base_w100_full_0.001lr_20260606_153534
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260610_130712
ait_temp_baseline_endtoend_b

In [70]:
# Load DPL dataset
test_data_dir = Path(f"../../experiments/{dataset}/{scenario}/{experiment}/deepproblog/cache/")
test_data_path = test_data_dir / "test_data.pkl"
test_data = pickle.load(open(test_data_path, "rb"))
test_data_df = pd.DataFrame(test_data)

In [71]:
def compute_weighted_f1(confusion_matrix, classes):
    # Calculate precision, recall, and F1 for each class
    precision = {}
    recall = {}
    f1 = {}
    support = {}

    for i, cls in enumerate(classes):
        tp = confusion_matrix[i][i]
        fp = sum(confusion_matrix[j][i] for j in range(len(classes)) if j != i)
        fn = sum(confusion_matrix[i][j] for j in range(len(classes)) if j != i)

        precision[cls] = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall[cls] = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1[cls] = 2 * (precision[cls] * recall[cls]) / (precision[cls] + recall[cls]) if (precision[cls] + recall[cls]) > 0 else 0
        support[cls] = tp + fn

    # Calculate weighted F1 score
    total_support = sum(support.values())
    weighted_f1 = sum(f1[cls] * support[cls] for cls in classes) / total_support if total_support > 0 else 0

    return weighted_f1

In [72]:
for exp_name, exp_data in dpl_experiments.items():
    experiment_name_full = exp_name
    experiment_name = experiment_name_full
    print(f"Processing {experiment_name}...")
    
    dpl_to_orig = dict(zip(test_data_df['dpl_index'], test_data_df['orig_index']))

    errors = exp_data["errors"]
    correct = exp_data["correct"]

    original_indices = []
    y_preds = []
    y_trues = []

    for err in errors:
        dpl_index = err['index']
        original_indices.append(dpl_to_orig[dpl_index])
        y_pred = err['predicted']
        y_true = err['actual']
        y_preds.append(phase_map[y_pred])
        y_trues.append(phase_map[y_true])

    for corr in correct:
        dpl_index = corr['index']
        original_indices.append(dpl_to_orig[dpl_index])
        y_pred = corr['predicted']
        y_true = corr['actual']
        y_preds.append(phase_map[y_pred])
        y_trues.append(phase_map[y_true])

    exp_df = df.loc[original_indices].copy()
    exp_df['y_pred'] = y_preds
    exp_df['y_true'] = y_trues

    metrics = exp_data["metrics"]
    macro_f1 = metrics["macro_f1"]
    cm = exp_data["confusion_matrix"]
    
    weighted_f1 = compute_weighted_f1(cm, metrics["classes"])

    dataframes, temp_metrics_dict = temp_metrics(exp_df, macro_f1, phase_starts)
    
    metrics_dict[experiment_name] = {
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": weighted_f1,
        "accuracy": metrics["accuracy"],
        "detection_rate": metrics["detection_rate"],
        "false_alarm_rate": metrics["false_alarm_rate"],
        "num_wrong": temp_metrics_dict["num_wrong"],
        "num_causal": temp_metrics_dict["num_causal"],
        "num_regression": temp_metrics_dict["num_regression"],
        "error_rate": temp_metrics_dict["error_rate"],
        "causal_rate": temp_metrics_dict["causal_rate"],
        "regression_rate": temp_metrics_dict["regression_rate"]
    }

    dataframes_dict[experiment_name] = dataframes

Processing ait_temp_context_pretrained_base_w100_full_0.001lr_20260605_215320...
Processing ait_temp_pretrained_base_w100_full_0.001lr_20260522_230036...
Processing ait_temp_endtoend_base_w100_full_0.001lr_20260606_015802...
Processing ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260613_093832...
Processing ait_temp_context_baseline_endtoend_base_w100_full_0.001lr_20260610_130712...
Processing ait_temp_context_endtoend_base_w100_full_0.001lr_20260605_124548...
Processing ait_temp_context_pretrained_base_w100_full_0.001lr_20260522_172417...
Processing ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260611_094556...
Processing ait_temp_baseline_endtoend_base_w100_full_0.001lr_20260524_230019...
Processing ait_temp_context_endtoend_base_w100_full_0.001lr_20260521_235742...
Processing ait_temp_endtoend_base_w100_full_0.001lr_20260606_153534...
Processing ait_temp_context_pretrained_base_w100_full_0.001lr_20260605_203228...
Processing ait_temp_pretrained_base_w100_full_0.001lr_20

### Baseline Models

In [73]:
# Load predictions 
pred_dir = experiment_dir / "baselines" / "predictions"
file_paths = list(pred_dir.iterdir())

predictions_dict = {}
for file_path in file_paths:
    experiment_name = file_path.stem
    print(f"Processing {experiment_name}...")

    predictions = np.load(file_path, allow_pickle=True)
    flow_idx = predictions["flow_idx"]
    y_true = predictions["y_true"]
    y_pred = predictions["y_pred"]

    predictions_dict[experiment_name] = {
        "flow_idx": flow_idx,
        "y_true": y_true,
        "y_pred": y_pred,
    }

Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260612_221820...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260609_223427...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260611_094802...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260610_010639...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260611_171427...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_x_x...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260610_022224...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_x_x...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260609_235014...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260611_234151...


In [74]:
# Load metrics and compute temporal consistency metrics
metrics_dir = experiment_dir / "baselines" / "metrics"
file_paths = list(metrics_dir.iterdir())

for file_path in file_paths:
    experiment_name = file_path.stem
    print(f"Processing {experiment_name}...")

    # Predictions
    pred = predictions_dict[experiment_name]
    flow_idx = pred["flow_idx"]
    y_true = pred["y_true"]
    y_pred = pred["y_pred"]

    # Metrics
    with open(file_path) as f:
        metrics = json.load(f)

    # Compute temporal consistency metrics
    exp_df = df.loc[flow_idx].copy()
    exp_df['y_pred'] = y_pred
    exp_df['y_true'] = y_true
    macro_f1 = metrics["macro_f1"]
    cm = metrics["confusion_matrix"]
    weighted_f1 = compute_weighted_f1(cm, metrics["classes"])

    dataframes, temp_metrics_dict = temp_metrics(exp_df, macro_f1, phase_starts)
    
    metrics_dict[experiment_name] = {
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": weighted_f1,
        "accuracy": metrics["accuracy"],
        "detection_rate": metrics["detection_rate"],
        "false_alarm_rate": metrics["false_alarm_rate"],
        "num_wrong": temp_metrics_dict["num_wrong"],
        "num_causal": temp_metrics_dict["num_causal"],
        "num_regression": temp_metrics_dict["num_regression"],
        "error_rate": temp_metrics_dict["error_rate"],
        "causal_rate": temp_metrics_dict["causal_rate"],
        "regression_rate": temp_metrics_dict["regression_rate"],
        "temp_score": temp_metrics_dict["temp_score"]
    }

    dataframes_dict[experiment_name] = dataframes

Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260611_094802...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_x_x...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260611_234151...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260612_221820...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260610_010639...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260609_223427...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_x_x...
Processing ensemble_basefeatures_w100_fulldata_0.001lr_20260611_171427...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260609_235014...
Processing multiclass_basefeatures_w100_fulldata_0.001lr_20260610_022224...


In [75]:
# Compute average metrics
avg_metrics = {}
for exp_name, metrics in metrics_dict.items():
    parts = exp_name.split("_")
    model_name = "_".join(parts[:-2])
    print(model_name)
    if model_name not in avg_metrics:
        avg_metrics[model_name] = {
            "macro_f1": [],
            "weighted_f1": [],
            "accuracy": [],
            "detection_rate": [],
            "false_alarm_rate": [],
            "num_wrong": [],
            "num_causal": [],
            "num_regression": [],
            "error_rate": [],
            "causal_rate": [],
            "regression_rate": [],
            "temp_score": [],
        }
    
    for metric_name, value in metrics.items():
        avg_metrics[model_name][metric_name].append(value)

ait_temp_context_pretrained_base_w100_full_0.001lr
ait_temp_pretrained_base_w100_full_0.001lr
ait_temp_endtoend_base_w100_full_0.001lr
ait_temp_baseline_endtoend_base_w100_full_0.001lr
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr
ait_temp_context_endtoend_base_w100_full_0.001lr
ait_temp_context_pretrained_base_w100_full_0.001lr
ait_temp_baseline_endtoend_base_w100_full_0.001lr
ait_temp_baseline_endtoend_base_w100_full_0.001lr
ait_temp_context_endtoend_base_w100_full_0.001lr
ait_temp_endtoend_base_w100_full_0.001lr
ait_temp_context_pretrained_base_w100_full_0.001lr
ait_temp_pretrained_base_w100_full_0.001lr
ait_temp_context_baseline_endtoend_base_w100_full_0.001lr
ait_temp_context_pretrained_base_w100_full_0.001lr
ait_temp_baseline_endtoend_base_w100_full_0.001lr
ait_temp_endtoend_base_w100_full_0.001lr
ait_temp_endtoend_base_w100_full_0.001lr
ait_temp_context_pretrained_base_w100_full_0.001lr
ait_temp_context_endtoend_base_w100_full_0.001lr
ait_temp_baseline_endtoend_base_

In [76]:
for model_name, metrics in avg_metrics.items():
    for metric_name, values in metrics.items():
        avg_metrics[model_name][metric_name] = {
            "mean": np.mean(values),
            "std": np.std(values)
        }

/home/sofia/Desktop/Thesis/DeepChronos/.venv/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/sofia/Desktop/Thesis/DeepChronos/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/sofia/Desktop/Thesis/DeepChronos/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sofia/Desktop/Thesis/DeepChronos/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/sofia/Desktop/Thesis/DeepChronos/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

In [77]:
avg_metrics_df = pd.DataFrame({
    "Model": list(avg_metrics.keys()),
    "Macro F1": [metrics["macro_f1"]["mean"] for metrics in avg_metrics.values()],
    "Macro F1 Std": [metrics["macro_f1"]["std"] for metrics in avg_metrics.values()],
    "Weighted F1": [metrics["weighted_f1"]["mean"] for metrics in avg_metrics.values()],
    "Weighted F1 Std": [metrics["weighted_f1"]["std"] for metrics in avg_metrics.values()],
    "Accuracy": [metrics["accuracy"]["mean"] for metrics in avg_metrics.values()],
    "Accuracy Std": [metrics["accuracy"]["std"] for metrics in avg_metrics.values()],
    "Detection Rate": [metrics["detection_rate"]["mean"] for metrics in avg_metrics.values()],
    "Detection Rate Std": [metrics["detection_rate"]["std"] for metrics in avg_metrics.values()],
    "False Alarm Rate": [metrics["false_alarm_rate"]["mean"] for metrics in avg_metrics.values()],
    "False Alarm Rate Std": [metrics["false_alarm_rate"]["std"] for metrics in avg_metrics.values()],
    "Causal Rate": [metrics["causal_rate"]["mean"] for metrics in avg_metrics.values()],
    "Causal Rate Std": [metrics["causal_rate"]["std"] for metrics in avg_metrics.values()],
    "Regression Rate": [metrics["regression_rate"]["mean"] for metrics in avg_metrics.values()],
    "Regression Rate Std": [metrics["regression_rate"]["std"] for metrics in avg_metrics.values()],
})

### Create metrics table

In [78]:
avg_metrics_df

,Model,Macro F1,Macro F1 Std,Weighted F1,Weighted F1 Std,Accuracy,Accuracy Std,Detection Rate,Detection Rate Std,False Alarm Rate,False Alarm Rate Std,Causal Rate,Causal Rate Std,Regression Rate,Regression Rate Std
0,ait_temp_context_pretrained_base_w100_full_0.0...,0.944343,0.023630,0.999176,0.000031,0.999172,0.000029,0.990414,0.000318,0.000335,0.000026,0.000000,0.000000,0.002198,0.004396
1,ait_temp_pretrained_base_w100_full_0.001lr,0.877630,0.020401,0.998392,0.000019,0.998381,0.000016,0.983372,0.001172,0.000775,0.000077,0.000000,0.000000,0.151707,0.045601
2,ait_temp_endtoend_base_w100_full_0.001lr,0.842047,0.013406,0.998234,0.000096,0.998215,0.000093,0.980827,0.002816,0.000807,0.000181,0.000000,0.000000,0.180134,0.046352
3,ait_temp_baseline_endtoend_base_w100_full_0.001lr,0.092314,0.000000,0.100002,0.000000,0.053244,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.199724,0.000000
4,ait_temp_context_baseline_endtoend_base_w100_f...,0.630656,0.038018,0.917392,0.051970,0.925371,0.030604,0.714113,0.335354,0.062748,0.051184,0.000000,0.000000,0.047424,0.038721
5,ait_temp_context_endtoend_base_w100_full_0.001lr,0.940046,0.010137,0.999072,0.000057,0.999070,0.000056,0.990875,0.001427,0.000469,0.000085,0.000000,0.000000,0.014495,0.010890
6,ensemble_basefeatures_w100_fulldata_0.001lr,0.858313,0.023975,0.997756,0.000284,0.997718,0.000302,0.991443,0.001444,0.001929,0.000327,0.026708,0.012588,0.017480,0.008151
7,multiclass_basefeatures_w100_fulldata_0.001lr,0.879975,0.014571,0.997696,0.000187,0.997671,0.000183,0.988212,0.001369,0.001797,0.000213,0.018880,0.010835,0.020534,0.013824


In [79]:
results_df_sorted_f1 = avg_metrics_df.sort_values("Macro F1", ascending=False)
results_df_sorted_f1

,Model,Macro F1,Macro F1 Std,Weighted F1,Weighted F1 Std,Accuracy,Accuracy Std,Detection Rate,Detection Rate Std,False Alarm Rate,False Alarm Rate Std,Causal Rate,Causal Rate Std,Regression Rate,Regression Rate Std
0,ait_temp_context_pretrained_base_w100_full_0.0...,0.944343,0.023630,0.999176,0.000031,0.999172,0.000029,0.990414,0.000318,0.000335,0.000026,0.000000,0.000000,0.002198,0.004396
5,ait_temp_context_endtoend_base_w100_full_0.001lr,0.940046,0.010137,0.999072,0.000057,0.999070,0.000056,0.990875,0.001427,0.000469,0.000085,0.000000,0.000000,0.014495,0.010890
7,multiclass_basefeatures_w100_fulldata_0.001lr,0.879975,0.014571,0.997696,0.000187,0.997671,0.000183,0.988212,0.001369,0.001797,0.000213,0.018880,0.010835,0.020534,0.013824
1,ait_temp_pretrained_base_w100_full_0.001lr,0.877630,0.020401,0.998392,0.000019,0.998381,0.000016,0.983372,0.001172,0.000775,0.000077,0.000000,0.000000,0.151707,0.045601
6,ensemble_basefeatures_w100_fulldata_0.001lr,0.858313,0.023975,0.997756,0.000284,0.997718,0.000302,0.991443,0.001444,0.001929,0.000327,0.026708,0.012588,0.017480,0.008151
2,ait_temp_endtoend_base_w100_full_0.001lr,0.842047,0.013406,0.998234,0.000096,0.998215,0.000093,0.980827,0.002816,0.000807,0.000181,0.000000,0.000000,0.180134,0.046352
4,ait_temp_context_baseline_endtoend_base_w100_f...,0.630656,0.038018,0.917392,0.051970,0.925371,0.030604,0.714113,0.335354,0.062748,0.051184,0.000000,0.000000,0.047424,0.038721
3,ait_temp_baseline_endtoend_base_w100_full_0.001lr,0.092314,0.000000,0.100002,0.000000,0.053244,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.199724,0.000000


In [80]:
# Save results
results_df_sorted_f1.to_csv(reports_dir / "avg_metrics.csv", index=False)

In [81]:
results_df_sorted_weighted = avg_metrics_df.sort_values("Weighted F1", ascending=False)
results_df_sorted_weighted

,Model,Macro F1,Macro F1 Std,Weighted F1,Weighted F1 Std,Accuracy,Accuracy Std,Detection Rate,Detection Rate Std,False Alarm Rate,False Alarm Rate Std,Causal Rate,Causal Rate Std,Regression Rate,Regression Rate Std
0,ait_temp_context_pretrained_base_w100_full_0.0...,0.944343,0.023630,0.999176,0.000031,0.999172,0.000029,0.990414,0.000318,0.000335,0.000026,0.000000,0.000000,0.002198,0.004396
5,ait_temp_context_endtoend_base_w100_full_0.001lr,0.940046,0.010137,0.999072,0.000057,0.999070,0.000056,0.990875,0.001427,0.000469,0.000085,0.000000,0.000000,0.014495,0.010890
1,ait_temp_pretrained_base_w100_full_0.001lr,0.877630,0.020401,0.998392,0.000019,0.998381,0.000016,0.983372,0.001172,0.000775,0.000077,0.000000,0.000000,0.151707,0.045601
2,ait_temp_endtoend_base_w100_full_0.001lr,0.842047,0.013406,0.998234,0.000096,0.998215,0.000093,0.980827,0.002816,0.000807,0.000181,0.000000,0.000000,0.180134,0.046352
6,ensemble_basefeatures_w100_fulldata_0.001lr,0.858313,0.023975,0.997756,0.000284,0.997718,0.000302,0.991443,0.001444,0.001929,0.000327,0.026708,0.012588,0.017480,0.008151
7,multiclass_basefeatures_w100_fulldata_0.001lr,0.879975,0.014571,0.997696,0.000187,0.997671,0.000183,0.988212,0.001369,0.001797,0.000213,0.018880,0.010835,0.020534,0.013824
4,ait_temp_context_baseline_endtoend_base_w100_f...,0.630656,0.038018,0.917392,0.051970,0.925371,0.030604,0.714113,0.335354,0.062748,0.051184,0.000000,0.000000,0.047424,0.038721
3,ait_temp_baseline_endtoend_base_w100_full_0.001lr,0.092314,0.000000,0.100002,0.000000,0.053244,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.199724,0.000000


## Plotting

In [82]:
plots_dir = reports_dir / "temp_plots"
plots_dir.mkdir(parents=True, exist_ok=True)

In [83]:
def sample_correct_predictions(correct_df, time_window="1T"):
    correct_attack_df = correct_df[correct_df["y_true"] != 0]
    correct_benign_df = correct_df[correct_df["y_true"] == 0]

    correct_attack_df_sampled = (
        correct_attack_df
        .set_index("start_time_dt")
        .resample(time_window)
        .first()
        .dropna()
        .reset_index()
    )
    correct_benign_df_sampled = (
        correct_benign_df
        .set_index("start_time_dt")
        .resample(time_window)
        .first()
        .dropna()
        .reset_index()
    )

    correct_df_sampled = pd.concat([correct_attack_df_sampled, correct_benign_df_sampled]).sort_values("start_time_dt")

    return correct_attack_df_sampled, correct_benign_df_sampled, correct_df_sampled

In [84]:
def plot_temp_consistency(
    df, 
    phase_bounds, 
    correct_df,
    causal_df,
    regression_df,
    metrics_dict,
    plot_name,
    exp_name, 
    labels,
    out_dir,
    attack_start=None, 
    attack_end=None,
    with_metrics_box=False,
    save_plot=True, 
    show_plot=True,
):
    
    df = df.copy()
    correct_df = correct_df.copy()
    causal_df = causal_df.copy()
    regression_df = regression_df.copy()

    # trim to specified attack window, if provided
    if attack_start is not None and attack_end is not None:
        df["start_time_dt"] = pd.to_datetime(df["start_time_dt"], errors="coerce")
        df["end_time_dt"]   = pd.to_datetime(df["end_time_dt"],   errors="coerce")
        df = df[(df["start_time_dt"] >= attack_start) & (df["end_time_dt"] <= attack_end)]
        
        correct_df   = correct_df[(correct_df["start_time_dt"] >= attack_start) & (correct_df["end_time_dt"] <= attack_end)]
        causal_df      = causal_df[(causal_df["start_time_dt"] >= attack_start) & (causal_df["end_time_dt"] <= attack_end)]
        regression_df  = regression_df[(regression_df["start_time_dt"] >= attack_start) & (regression_df["end_time_dt"] <= attack_end)]

    fig, ax = plt.subplots(figsize=(14, 5))

    edges  = phase_bounds["min"].tolist() + [phase_bounds["max"].iloc[-1]]
    values = phase_bounds.index.tolist()

    ax.stairs(
        values=values,
        edges=edges,
        linewidth=2,
        color="black",
        label="True phase"
    )

    values.append(5)
    ax.fill_between(
        edges,                    
        values,              
        [min(values) - 0.5]*len(values),  
        step="post",
        color="gray",
        alpha=0.2
    )

    ax.scatter(correct_df["start_time_dt"], correct_df["y_pred"], s=25, marker="o",
            color="green", alpha=0.5, label="Correct predictions")

    ax.scatter(causal_df["start_time_dt"], causal_df["y_pred"], s=90, marker="X",
               color="red", alpha=0.9, label="Causal violations")

    ax.scatter(regression_df["start_time_dt"], regression_df["y_pred"], s=90, marker="X",
               color="orange", alpha=0.9, label="Regression violations")

    ax.set_ylim(-0.5, max(values) + 2)
    ax.set_xlim(df["start_time_dt"].min(), df["start_time_dt"].max())
    ax.set_xlabel("Time")
    ax.set_ylabel("Phase")
    ax.set_title(plot_name, fontsize=16)
    ytick_positions = [i for i, _ in enumerate(labels)]
    ax.set_yticks(sorted(ytick_positions))
    ax.set_yticklabels(labels)
    ax.grid(alpha=0.2)

    if with_metrics_box:
        textstr = (
            f"# Total violations: {metrics_dict['num_wrong']}\n"
            f"# Causal violations: {metrics_dict['num_causal']}\n"
            f"# Regression violations: {metrics_dict['num_regression']}\n"
        )

        ax.text(
            0.02, 0.95, textstr,
            transform=ax.transAxes,
            fontsize=12,
            verticalalignment='top',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7)
        )
    
    ax.legend(
        frameon=True,
        facecolor="white",
        framealpha=0.9,
        loc="upper right",
        prop={"size": 12}
    )

    fig.tight_layout()

    if save_plot:
        out_path = out_dir / f"{exp_name}.png"
        print(f"Saving plot to {out_path}...")
        fig.savefig(out_path, dpi=300, bbox_inches="tight")

    if show_plot:
        plt.show()

### DPL Models

In [85]:
def get_plot_name(experiment_name):
    # DPL Model Names
    if "temp_context" in experiment_name:
        model_name = "Temporal and Attack-Phase Logic"
    elif "temp" in experiment_name:
        model_name = "Temporal Logic"
    elif "context" in experiment_name:
        model_name = "Attack-Phase Logic"

    if "baseline" in experiment_name:
        plot_name = f"Logic Baseline - {model_name}"
    elif "pretrained" in experiment_name:
        plot_name = f"DPL Pretrained - {model_name}"
    elif "endtoend" in experiment_name:
        plot_name = f"DPL End-to-End - {model_name}"

    # Neural Baseline Names
    if "multiclass" in experiment_name:
        plot_name = "Neural Baseline - Multiclass LSTM"
    elif "ensemble" in experiment_name:
        plot_name = "Neural Baseline - LSTM Ensemble"
    
    return plot_name

In [86]:
# for experiment_name, metrics in metrics_dict.items():
#     print(f"Processing {experiment_name}...")
    
#     plot_name = get_plot_name(experiment_name)

#     dataframes = dataframes_dict[experiment_name]
#     correct_df = dataframes["correct"]
#     causal_df = dataframes["causal"]
#     regression_df = dataframes["regression"]

#     # Plot full window
#     correct_attack_df_sampled, correct_benign_df_sampled, correct_df_sampled = \
#         sample_correct_predictions(correct_df, time_window="3min")
    
#     # start = pd.Timestamp("2022-01-14 12:00")
#     # end   = pd.Timestamp("2022-01-17 13:30")
    
#     plot_temp_consistency(
#         df=df,
#         phase_bounds=phase_bounds, 
#         correct_df=correct_df_sampled,
#         causal_df=causal_df,
#         regression_df=regression_df,
#         metrics_dict=metrics,
#         plot_name=plot_name,
#         exp_name=f"{experiment_name}_full_window",
#         labels=plot_labels,
#         out_dir=plots_dir,
#         # attack_start=start,
#         # attack_end=end,
#         with_metrics_box=True,
#         save_plot=True,
#         show_plot=True
#     )

#     if dataset == "aitv2":
#         # Zoom in on attack window
#         correct_attack_df_sampled, correct_benign_df_sampled, correct_df_sampled = \
#             sample_correct_predictions(correct_df, time_window="1min")

#         start = attack_start
#         end   = pd.Timestamp("2022-01-17 12:05")
        
#         plot_temp_consistency(
#             df=df,
#             phase_bounds=phase_bounds, 
#             correct_df=correct_df_sampled,
#             causal_df=causal_df,
#             regression_df=regression_df,
#             metrics_dict=metrics,
#             plot_name=plot_name, 
#             exp_name=f"{experiment_name}_zoomed_in",
#             labels=plot_labels,
#             out_dir=plots_dir,
#             attack_start=start,
#             attack_end=end,
#             with_metrics_box=True,
#             save_plot=True,
#             show_plot=True
#         )